# 新鮮 / 腐爛水果分類（PyTorch CNN）
資料集：Kaggle `sriramr/fruits-fresh-and-rotten-for-classification`（6 類）

流程：資料處理 → 模型訓練（自建 CNN vs ResNet18 遷移學習）→ 結果評估 → Demo

> 執行前：執行階段 → 變更執行階段類型 → **T4 GPU**

In [ ]:
!nvidia-smi -L

## 0. 取得程式碼
從 GitHub clone 專案程式碼

In [ ]:
!git clone https://github.com/jerry910127/-.git fruit_project
%cd fruit_project

## 1. 下載資料集

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
print(DATA_DIR)

## 2. 資料探索（EDA）

In [ ]:
from IPython.display import Image, display
!python eda.py --data_dir {DATA_DIR}
for f in ["eda_class_counts", "eda_samples", "eda_sizes"]:
    display(Image(f"result/{f}.png", width=700))

## 3-A. 訓練自建 CNN（baseline）

In [ ]:
!python train.py --data_dir {DATA_DIR} --arch cnn --epochs 15
display(Image("result/curve_cnn.png"))

## 3-B. 訓練 ResNet18（ImageNet 預訓練，遷移學習）

In [ ]:
!python train.py --data_dir {DATA_DIR} --arch resnet18 --epochs 8
display(Image("result/curve_resnet18.png"))

## 4. Test set 評估

In [ ]:
for arch in ["cnn", "resnet18"]:
    !python evaluate.py --data_dir {DATA_DIR} --ckpt result/best_{arch}.pth
    display(Image(f"result/confusion_{arch}.png", width=550))
    display(Image(f"result/misclassified_{arch}.png", width=700))

In [ ]:
import pandas as pd
pd.read_csv("result/summary.csv")

## 5. Demo：上傳自己拍的水果照片

In [ ]:
from google.colab import files
import os
os.makedirs("demo", exist_ok=True)
for name, data in files.upload().items():
    open(f"demo/{name}", "wb").write(data)
!python predict.py --ckpt result/best_resnet18.pth --out result/predict.png demo
display(Image("result/predict.png"))

## （報告前）把訓練結果存到 Google Drive，現場直接載入、不必重訓

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# 備份：
!mkdir -p /content/drive/MyDrive/fruit_result && cp -r result/* /content/drive/MyDrive/fruit_result/
# 還原（報告當天，跳過 3-A / 3-B 直接跑 4、5）：
# !mkdir -p result && cp -r /content/drive/MyDrive/fruit_result/* result/